In [2]:
# ============================================================
# CS 340 PROJECT TWO
# GRAZIOSO SALVARE ANIMAL RESCUE DASHBOARD
# Developer: Kimberly Ceballos
# ============================================================

# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

from jupyter_dash import JupyterDash

import dash_leaflet as dl

from dash import dcc, html, dash_table

from dash.dependencies import Input, Output

import plotly.express as px

import pandas as pd

import numpy as np

import base64

import os

import logging

# Import MongoDB CRUD module
from animal_shelter import AnimalShelter


# ============================================================
# CONFIGURE JUPYTER DASH
# ============================================================

JupyterDash.infer_jupyter_proxy_config()

logging.basicConfig(level=logging.INFO)

LOGGER = logging.getLogger(__name__)


# ============================================================
# DATABASE CONNECTION / MODEL
# ============================================================

username = "aacuser"
password = os.getenv("AAC_PASSWORD", "SNHU1234")

# Connect to MongoDB
db = AnimalShelter(username, password)

# Retrieve all records from MongoDB
records = db.read({})

# Convert MongoDB records to Pandas DataFrame
df = pd.DataFrame.from_records(records)

# Remove MongoDB ObjectID column
df.drop(
    columns=["_id"],
    inplace=True,
    errors="ignore"
)

# Verify database contains records
if df.empty:
    print("WARNING: No records were retrieved from MongoDB.")
    print("Check your database connection and collection.")

else:
    print("MongoDB connection successful.")
    print("Total animal records:", len(df))
    print("Available columns:", list(df.columns))


# ============================================================
# RESCUE FILTER QUERIES
# ============================================================

def get_filter_query(filter_type):
    """
    Return MongoDB queries based on Grazioso Salvare's
    required rescue training criteria.
    """

    # --------------------------------------------------------
    # WATER RESCUE
    # --------------------------------------------------------

    if filter_type == "Water Rescue":

        return {
            "animal_type": "Dog",

            "breed": {
                "$regex": (
                    "Labrador Retriever|"
                    "Chesapeake Bay Retriever|"
                    "Newfoundland"
                ),
                "$options": "i"
            },

            "sex_upon_outcome": "Intact Female",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # --------------------------------------------------------
    # MOUNTAIN OR WILDERNESS RESCUE
    # --------------------------------------------------------

    elif filter_type == "Mountain or Wilderness Rescue":

        return {
            "animal_type": "Dog",

            "breed": {
                "$regex": (
                    "German Shepherd|"
                    "Alaskan Malamute|"
                    "Old English Sheepdog|"
                    "Siberian Husky|"
                    "Rottweiler"
                ),
                "$options": "i"
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # --------------------------------------------------------
    # DISASTER OR INDIVIDUAL TRACKING
    # --------------------------------------------------------

    elif filter_type == "Disaster or Individual Tracking":

        return {
            "animal_type": "Dog",

            "breed": {
                "$regex": (
                    "Doberman Pinscher|"
                    "German Shepherd|"
                    "Golden Retriever|"
                    "Bloodhound|"
                    "Rottweiler"
                ),
                "$options": "i"
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    # --------------------------------------------------------
    # RESET / SHOW ALL RECORDS
    # --------------------------------------------------------

    else:
        return {}


# ============================================================
# INITIALIZE DASHBOARD
# ============================================================

app = JupyterDash(__name__)

app.title = "Grazioso Salvare Animal Rescue Dashboard"


# ============================================================
# LOAD GRAZIOSO SALVARE LOGO
# ============================================================

image_filename = "Grazioso Salvare Logo.png"

if os.path.exists(image_filename):

    with open(image_filename, "rb") as image_file:

        encoded_image = base64.b64encode(
            image_file.read()
        ).decode("utf-8")

    logo = html.A(

        html.Img(

            src="data:image/png;base64,{}".format(
                encoded_image
            ),

            style={
                "height": "120px",
                "maxWidth": "100%",
                "objectFit": "contain",
                "margin": "10px"
            }

        ),

        href="https://www.snhu.edu",

        target="_blank"

    )

else:

    logo = html.A(

        html.H2("Grazioso Salvare"),

        href="https://www.snhu.edu",

        target="_blank"

    )

    print(
        "Logo not found. Place Grazioso Salvare Logo.png "
        "in the same directory as the notebook."
    )


# ============================================================
# DASHBOARD LAYOUT / VIEW
# ============================================================

app.layout = html.Div(

    [

        # ----------------------------------------------------
        # DASHBOARD HEADER
        # ----------------------------------------------------

        html.Center(

            [

                logo,

                html.H1(

                    "Grazioso Salvare Animal Rescue Dashboard",

                    style={
                        "color": "#003366",
                        "fontFamily": "Arial",
                        "marginBottom": "10px"
                    }

                ),

                html.H3(

                    "Developed by Kimberly Ceballos",

                    style={
                        "color": "#555555",
                        "fontFamily": "Arial"
                    }

                )

            ]

        ),

        html.Hr(),

        # ----------------------------------------------------
        # INTERACTIVE FILTER OPTIONS
        # ----------------------------------------------------

        html.Div(

            [

                html.H3(

                    "Select Rescue Training Type",

                    style={
                        "color": "#003366"
                    }

                ),

                dcc.RadioItems(

                    id="filter-type",

                    options=[

                        {
                            "label": "Water Rescue",
                            "value": "Water Rescue"
                        },

                        {
                            "label": "Mountain or Wilderness Rescue",
                            "value": "Mountain or Wilderness Rescue"
                        },

                        {
                            "label": "Disaster or Individual Tracking",
                            "value": "Disaster or Individual Tracking"
                        },

                        {
                            "label": "Reset",
                            "value": "Reset"
                        }

                    ],

                    value="Reset",

                    labelStyle={
                        "display": "inline-block",
                        "marginRight": "25px",
                        "marginBottom": "10px",
                        "cursor": "pointer"
                    },

                    inputStyle={
                        "marginRight": "8px"
                    }

                )

            ],

            style={
                "textAlign": "center",
                "padding": "20px",
                "backgroundColor": "#F2F6FA",
                "borderRadius": "8px"
            }

        ),

        html.Hr(),

        # ----------------------------------------------------
        # RECORD COUNT
        # ----------------------------------------------------

        html.Div(

            id="record-count",

            style={
                "textAlign": "center",
                "fontSize": "18px",
                "fontWeight": "bold",
                "color": "#003366",
                "marginBottom": "15px"
            }

        ),

        # ----------------------------------------------------
        # INTERACTIVE DATA TABLE
        # ----------------------------------------------------

        dash_table.DataTable(

            id="datatable-id",

            columns=[

                {
                    "name": column,
                    "id": column,
                    "deletable": False,
                    "selectable": True
                }

                for column in df.columns

            ],

            data=df.to_dict("records"),

            page_size=10,

            page_action="native",

            sort_action="native",

            sort_mode="multi",

            filter_action="native",

            row_selectable="single",

            selected_rows=[0] if not df.empty else [],

            selected_columns=[],

            style_table={
                "overflowX": "auto",
                "width": "100%"
            },

            style_cell={
                "textAlign": "left",
                "padding": "10px",
                "minWidth": "100px",
                "maxWidth": "250px",
                "overflow": "hidden",
                "textOverflow": "ellipsis",
                "fontFamily": "Arial",
                "fontSize": "13px"
            },

            style_header={
                "backgroundColor": "#003366",
                "color": "white",
                "fontWeight": "bold",
                "textAlign": "center"
            },

            style_data={
                "backgroundColor": "#FFFFFF",
                "color": "#222222"
            },

            style_data_conditional=[

                {
                    "if": {
                        "row_index": "odd"
                    },
                    "backgroundColor": "#F2F6FA"
                }

            ]

        ),

        html.Br(),

        html.Hr(),

        # ----------------------------------------------------
        # DASHBOARD CHARTS
        # ----------------------------------------------------

        html.Div(

            className="row",

            style={
                "display": "flex",
                "flexWrap": "wrap",
                "justifyContent": "space-between",
                "gap": "20px"
            },

            children=[

                # Breed distribution chart
                html.Div(

                    id="graph-id",

                    style={
                        "flex": "1",
                        "minWidth": "400px"
                    }

                ),

                # Geolocation map
                html.Div(

                    id="map-id",

                    style={
                        "flex": "1",
                        "minWidth": "400px"
                    }

                )

            ]

        ),

        html.Hr(),

        # ----------------------------------------------------
        # DASHBOARD FOOTER
        # ----------------------------------------------------

        html.Center(

            html.P(

                "CS 340 Project Two | Grazioso Salvare | "
                "Kimberly Ceballos",

                style={
                    "fontSize": "13px",
                    "color": "#666666"
                }

            )

        )

    ],

    style={
        "padding": "20px",
        "fontFamily": "Arial",
        "backgroundColor": "#FFFFFF"
    }

)


# ============================================================
# CONTROLLER / CALLBACKS
# ============================================================


# ============================================================
# CALLBACK 1: UPDATE DATA TABLE
# ============================================================

@app.callback(

    [
        Output("datatable-id", "data"),
        Output("datatable-id", "selected_rows"),
        Output("record-count", "children")
    ],

    Input("filter-type", "value")

)
def update_dashboard(filter_type):
    """
    Retrieve records from MongoDB based on the selected
    rescue training category.
    """

    query = get_filter_query(filter_type)

    try:

        filtered_records = db.read(query)

        filtered_df = pd.DataFrame.from_records(
            filtered_records
        )

        # Remove MongoDB ObjectID
        filtered_df.drop(
            columns=["_id"],
            inplace=True,
            errors="ignore"
        )

        # Match original table columns
        filtered_df = filtered_df.reindex(
            columns=df.columns
        )

        # Convert to dictionary records
        data = filtered_df.to_dict("records")

        # Select first record if results exist
        selected_rows = [0] if data else []

        record_count = (
            f"Showing {len(data)} animal records "
            f"for {filter_type}"
        )

        return data, selected_rows, record_count

    except Exception:

        LOGGER.exception("Dashboard filtering failed")

        return [], [], "Unable to retrieve animal records."


# ============================================================
# CALLBACK 2: UPDATE BREED DISTRIBUTION PIE CHART
# ============================================================

@app.callback(

    Output("graph-id", "children"),

    Input("datatable-id", "derived_virtual_data")

)
def update_graphs(viewData):
    """
    Update the breed distribution chart based on
    currently filtered data table records.
    """

    if viewData is None:

        dff = df.copy()

    else:

        dff = pd.DataFrame(viewData)

    if dff.empty or "breed" not in dff.columns:

        return html.Div(

            "No animal records available for this filter.",

            style={
                "textAlign": "center",
                "padding": "30px",
                "color": "#666666"
            }

        )

    # Count animals by breed
    breed_counts = (

        dff["breed"]

        .fillna("Unknown")

        .value_counts()

        .head(10)

        .reset_index()

    )

    breed_counts.columns = [
        "breed",
        "count"
    ]

    # Create pie chart
    fig = px.pie(

        breed_counts,

        names="breed",

        values="count",

        title="Top 10 Animal Breeds"

    )

    fig.update_traces(

        textposition="inside",

        textinfo="percent"

    )

    fig.update_layout(

        margin=dict(
            l=20,
            r=20,
            t=50,
            b=20
        ),

        legend_title_text="Animal Breed"

    )

    return [

        dcc.Graph(

            figure=fig,

            style={
                "width": "100%",
                "height": "500px"
            }

        )

    ]


# ============================================================
# CALLBACK 3: HIGHLIGHT SELECTED TABLE COLUMNS
# ============================================================

@app.callback(

    Output(
        "datatable-id",
        "style_data_conditional"
    ),

    Input(
        "datatable-id",
        "selected_columns"
    )

)
def update_styles(selected_columns):
    """
    Highlight selected columns in the data table.
    """

    styles = [

        {
            "if": {
                "row_index": "odd"
            },
            "backgroundColor": "#F2F6FA"
        }

    ]

    if selected_columns is None:

        return styles

    for column in selected_columns:

        styles.append(

            {
                "if": {
                    "column_id": column
                },
                "backgroundColor": "#D2F3FF"
            }

        )

    return styles


# ============================================================
# CALLBACK 4: UPDATE GEOLOCATION MAP
# ============================================================

@app.callback(

    Output("map-id", "children"),

    [
        Input(
            "datatable-id",
            "derived_virtual_data"
        ),

        Input(
            "datatable-id",
            "derived_virtual_selected_rows"
        )
    ]

)
def update_map(viewData, selected_rows):
    """
    Update geolocation map based on the filtered
    data and selected animal.
    """

    if viewData is None:

        dff = df.copy()

    else:

        dff = pd.DataFrame(viewData)

    # Check for empty data
    if dff.empty:

        return html.Div(

            "No location data available.",

            style={
                "textAlign": "center",
                "padding": "30px"
            }

        )

    # Ensure required coordinate columns exist
    required_columns = [
        "location_lat",
        "location_long"
    ]

    if not all(
        column in dff.columns
        for column in required_columns
    ):

        return html.Div(
            "Location columns are missing from the dataset."
        )

    # Determine selected row
    row = 0

    if selected_rows is not None and len(selected_rows) > 0:

        selected_index = selected_rows[0]

        if 0 <= selected_index < len(dff):

            row = selected_index

    # Get selected animal
    animal = dff.iloc[row]

    latitude = pd.to_numeric(

        animal.get("location_lat"),

        errors="coerce"

    )

    longitude = pd.to_numeric(

        animal.get("location_long"),

        errors="coerce"

    )

    # Handle invalid coordinates
    if pd.isna(latitude) or pd.isna(longitude):

        return html.Div(

            "Location unavailable for selected animal.",

            style={
                "textAlign": "center",
                "padding": "30px"
            }

        )

    # Animal details
    animal_name = str(
        animal.get("name", "Unknown")
    )

    animal_breed = str(
        animal.get("breed", "Unknown")
    )

    animal_type = str(
        animal.get("animal_type", "Unknown")
    )

    # Create geolocation map
    return [

        dl.Map(

            style={
                "width": "100%",
                "height": "500px"
            },

            center=[
                float(latitude),
                float(longitude)
            ],

            zoom=10,

            children=[

                dl.TileLayer(),

                dl.Marker(

                    position=[
                        float(latitude),
                        float(longitude)
                    ],

                    children=[

                        dl.Tooltip(
                            animal_breed
                        ),

                        dl.Popup(

                            [

                                html.H3(
                                    "Animal Information"
                                ),

                                html.P(
                                    "Name: " + animal_name
                                ),

                                html.P(
                                    "Breed: " + animal_breed
                                ),

                                html.P(
                                    "Animal Type: " + animal_type
                                )

                            ]

                        )

                    ]

                )

            ]

        )

    ]


# ============================================================
# RUN DASHBOARD
# ============================================================

print("Starting Grazioso Salvare Dashboard...")

app.run_server(
    mode="inline",
    debug=False,
    port=8050
)

In [ ]:
from pymongo import MongoClient

try:
    client = MongoClient(
        "mongodb://localhost:27017/",
        username="aacuser",
        password="SNHU1234",
        authSource="aac",
        serverSelectionTimeoutMS=5000
    )

    client.admin.command("ping")

    print("MongoDB connection successful!")

    database = client["aac"]
    collection = database["animals"]

    print("Total records:", collection.count_documents({}))

except Exception as e:
    print("MongoDB connection failed!")
    print(type(e).__name__)
    print(str(e))

In [ ]:
from animal_shelter import AnimalShelter

db = AnimalShelter(
    username="aacuser",
    password="SNHU1234"
)

if db.test_connection():
    records = db.read({})
    print("Total animal records:", len(records))
else:
    print("Check your MongoDB credentials and authentication database.")


In [ ]:
from animal_shelter import AnimalShelter

username = "aacuser"
password = "SNHU1234"

for auth_source in ["aac", "admin"]:
    print(f"\nTesting authentication database: {auth_source}")

    db = AnimalShelter(
        username=username,
        password=password,
        auth_source=auth_source
    )

    if db.test_connection():
        print(f"SUCCESS: {auth_source}")
        print("Animal records:", len(db.read({})))
    else:
        print(f"FAILED: {auth_source}")

    db.close()

In [ ]:
from animal_shelter import AnimalShelter

db = AnimalShelter(
    username="aacuser",
    password="SNHU1234",
    auth_source="aac"
)

if db.test_connection():

    print("SUCCESS! MongoDB is connected.")

    records = db.read({})

    print("Total animal records:", len(records))

else:

    print("MongoDB authentication still failed.")

In [ ]:
from animal_shelter import AnimalShelter

db = AnimalShelter(
    username="aacuser",
    password="SNHU1234",
    auth_source="aac"
)

if db.test_connection():

    print("SUCCESS! MongoDB is connected.")

    records = db.read({})

    print("Total animal records:", len(records))

else:

    print("MongoDB authentication still failed.")

In [ ]:
from animal_shelter import AnimalShelter

db = AnimalShelter(
    username="aacuser",
    password="SNHU1234",
    auth_source="aac"
)

if db.test_connection():
    print("SUCCESS! MongoDB connected!")

    records = db.read({})

    print("Total animal records:", len(records))

    if records:
        print("First animal:", records[0].get("name", "Unknown"))
else:
    print("MongoDB connection failed.")

/home/codio/.pyenv/versions/3.11.2/lib/python3.11/site-packages/jupyter_dash/comms.py:100: RuntimeWarning: coroutine 'Kernel.execute_request' was never awaited
  kernel.execute_request(stream, ident, parent)


MongoDB connection successful.
Total animal records: 50001
Available columns: ['rec_num', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed', 'color', 'date_of_birth', 'datetime', 'monthyear', 'name', 'outcome_subtype', 'outcome_type', 'sex_upon_outcome', 'location_lat', 'location_long', 'age_upon_outcome_in_weeks', 'is_test_record']


INFO:werkzeug: * Running on http://127.0.0.1:8050/ (Press CTRL+C to quit)
INFO:werkzeug:127.0.0.1 - - [19/Oct/2026 05:16:32] "GET /_alive_a6e637e1-c032-481d-90f1-d54f39463343 HTTP/1.1" 200 -


Starting Grazioso Salvare Dashboard...


INFO:werkzeug:127.0.0.1 - - [19/Oct/2026 05:16:32] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Oct/2026 05:16:33] "GET /_dash-dependencies HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Oct/2026 05:16:34] "GET /_dash-layout HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Oct/2026 05:16:34] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [19/Oct/2026 05:16:34] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [19/Oct/2026 05:16:34] "POST /_dash-update-component HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Oct/2026 05:16:34] "POST /_dash-update-component HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Oct/2026 05:16:38] "POST /_dash-update-component HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Oct/2026 05:16:39] "POST /_dash-update-component HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Oct/2026 05:16:41] "POST /_dash-update-component HTTP/1.1" 200 -
INFO: